# img2txt: Feature Extraction & Analysis Pipeline

## Updates

This notebook has been refactored to use modular feature extraction APIs:

### New Modules
- **`feature_extraction_batch.py`** - For extracting features from entire datasets
- **`feature_extraction_inference.py`** - For single-image analysis

### Workflow
1. **Import modules** - Load feature extraction APIs
2. **Extract features** - Use batch or inference API
3. **Bucket features** - Organize by category (Step 2)
4. **Train importance model** - Learn which features matter (Step 3)
5. **Generate descriptions** - LLM creates text from features (Step 4)

### APIs

#### Batch Processing (for datasets)
```python
df = extract_features_batch(
    image_dir='path/to/images/',
    metadata_csv='metadata.csv',
    yolo_weights='yolo.pt',
    unet_weights='unet.pth',
)
```

#### Inference (for single images)
```python
result = extract_features_and_rank(
    image_path='lesion.jpg',
    yolo_weights='yolo.pt',
    unet_weights='unet.pth',
    importance_model_path='model.pth',  # optional
)
```

See CLAUDE.md for full documentation.

In [1]:
!pip install python-resize-image -q
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.3 MB/s eta 0:00:0000:01


In [ ]:
"""Step 1: Import feature extraction modules
These modules handle:
- Automatic lesion segmentation (YOLO + UNet fallback)
- Feature extraction (color, shape, border, texture)
- Dataset building (batch) or single-image analysis (inference)
"""

import numpy as np
import pandas as pd
import json
from pathlib import Path
from tqdm import tqdm
import torch

# Feature extraction APIs
from extraction.feature_extraction_batch import extract_features_batch
from extraction.feature_extraction_inference import extract_features_and_rank
from analysis.threshold_rules import row_to_labels
from config.config import FEATURE_ROUTING, _safe_number

In [ ]:
"""Step 1a: Extract features from all images (BATCH)

If you have a directory of images and a CSV with metadata,
use extract_features_batch() to process everything at once.
"""

# Configuration
META_CSV = "/kaggle/input/datasets/mihailodin1/classification-results/classification_results.csv"
IMAGE_DIR = "/kaggle/input/datasets/mihailodin1/all-image-skin"
YOLO_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_yolo.pt"
UNET_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_unet.pth"
OUTPUT_CSV = "features_dataset.csv"

# Uncomment to extract features:
# df = extract_features_batch(
#     image_dir=IMAGE_DIR,
#     metadata_csv=META_CSV,
#     yolo_weights=YOLO_WEIGHTS,
#     unet_weights=UNET_WEIGHTS,
#     verbose=True,
# )
# df.to_csv(OUTPUT_CSV, index=False)
# print(f"Saved {len(df)} images")

## Stage1. Извлечение признаков из дерматоскопических изображений. Подготовка датасета

In [ ]:
# Load pre-computed features from CSV
df = pd.read_csv("/kaggle/input/datasets/mihailodin1/features-img2txt/features_dataset.csv")
print(f"Loaded {len(df)} images with features")

In [ ]:
"""Step 1b: Extract features from a single image (INFERENCE)

For analyzing one lesion image (e.g., in Step 4 for LLM text generation).
"""

def extract_single_lesion(image_path, importance_model_path=None):
    """Extract features and optionally rank by importance."""
    result = extract_features_and_rank(
        image_path=image_path,
        yolo_weights=YOLO_WEIGHTS,
        unet_weights=UNET_WEIGHTS,
        importance_model_path=importance_model_path,
    )
    return result

# Example:
# result = extract_single_lesion('test_image.jpg')
# print("Features:", result['features_dict'])
# if 'important_features' in result:
#     print("Top-10 important features:", result['important_features'])

In [ ]:
"""Step 2: Explore feature structure"""

# Show feature categories
feature_categories = {}
for feature_name, (category_path, description, unit) in FEATURE_ROUTING.items():
    category = category_path.split('.')[0]
    if category not in feature_categories:
        feature_categories[category] = []
    feature_categories[category].append(feature_name)

for cat, features in sorted(feature_categories.items()):
    print(f"{cat}: {len(features)} features")

print(f"\nTotal features: {len(FEATURE_ROUTING)}")

## Stage 2. Бакетирование признаков.

In [ ]:
"""Step 2: Feature Bucketing

Convert numeric features into categorical labels using threshold rules.
Thresholds are defined in threshold_rules.py and threshold_config.py
"""

from analysis.feature_bucketing_batch import bucket_features_batch, get_label_statistics
from analysis.feature_bucketing_inference import bucket_features

# Apply bucketing to entire dataset
df = bucket_features_batch(df, verbose=True)

print(f"\nDataset after bucketing:")
print(f"  Rows: {len(df)}")
print(f"  Columns: {df.shape[1]}")
print(f"  New columns: features_organized, labels, labels_json")

In [ ]:
"""Step 2b: Bucketing for a single image (Inference)

If you extracted features for one image in Step 1, bucket them here.
"""

def bucket_single_lesion(features_dict):
    """Bucket features for a single image."""
    result = bucket_features(features_dict)
    return result

# Example:
# If you have features from extract_single_lesion():
# features_dict = result['features_dict']
# bucketed = bucket_single_lesion(features_dict)
# print("Labels:", bucketed['labels'])
# print("Features organized:", bucketed['features_json'])

In [ ]:
"""Explore label distribution in dataset"""

# Check what labels were generated
if 'labels' in df.columns:
    print("Label examples (first 3 rows):")
    for i in range(min(3, len(df))):
        labels = df.iloc[i]['labels']
        print(f"\n  Image {i}:")
        if isinstance(labels, dict):
            for key, val in labels.items():
                print(f"    {key}: {val}")

# Get statistics on labels
stats = get_label_statistics(df)
if not stats.empty:
    print(f"\nLabel statistics:")
    print(stats.to_string())

## Stage 3. Модель отбора важных признаков.

In [ ]:
from importance.importance_training_batch import generate_pseudo_labels, train_importance_model, prepare_for_training
from importance.importance_ranking_inference import rank_features, rank_features_batch

### Pseudo-label generation (development only)

In [ ]:
# Generate pseudo-labels from statistical and metadata priors
df = generate_pseudo_labels(
    df,
    labels_col="labels",
    features_col="features_json",
    top_k=15,
    mode="combined",
    z_weight=1.5,
    prior_weight=0.8,
)

# Copy to important_labels for training
df["important_labels"] = df["pseudo_important_labels"]

print("Example pseudo_important_labels:")
print(df["pseudo_important_labels"].iloc[0])
print("\nColumn important_labels is ready for training")

### Model training (development only)

In [ ]:
# Save dataset for training
train_csv = prepare_for_training(
    df,
    csv_path="features_dataset_for_importance.csv",
    labels_col="labels",
    top_k=15,
    z_weight=1.5,
    prior_weight=0.8,
)

# Train importance model
best_score = train_importance_model(
    data_csv=train_csv,
    image_dir=ROOT_DIR,
    epochs=70,
    batch_size=64,
    lr=1e-4,
    out_dir="importance_checkpoints",
)

print(f"Best validation score: {best_score:.4f}")

### Feature ranking with pre-trained model

In [ ]:
# Rank features for single image using trained model
def rank_single_image(image_path, model_checkpoint=None):
    """Rank features for a single image."""
    import torch
    from extraction.feature_extraction_inference import extract_features_and_rank
    from analysis.feature_bucketing_inference import bucket_features
    from importance.importance_ranking_inference import rank_features
    
    # Extract features from image
    extraction = extract_features_and_rank(image_path)
    features_dict = extraction['features_dict']
    image = extraction.get('image') if 'image' in extraction else None
    
    # Bucket features into labels
    bucketed = bucket_features(features_dict)
    labels = bucketed['labels']
    
    # Rank by importance if model provided
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ranking = rank_features(
        features_dict,
        labels,
        image=image if model_checkpoint else None,
        importance_model_path=model_checkpoint,
        device=device,
    )
    
    return {
        'features': features_dict,
        'labels': labels,
        'important_features': ranking['important_features'],
    }

# Example: rank features for a sample image
# model_path = "importance_checkpoints/best.pt"  # Set if you have a trained model
# result = rank_single_image("path/to/image.jpg", model_checkpoint=model_path)
# print("Important features:", result['important_features'])

In [ ]:
# Rank features for entire dataset
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Option 1: With trained model (if available)
# df_ranked = rank_features_batch(
#     df,
#     importance_model_path="importance_checkpoints/best.pt",
#     image_dir=ROOT_DIR,
#     device=device,
#     verbose=True,
# )

# Option 2: Without model (important_labels will be empty)
df_ranked = rank_features_batch(df, verbose=True)

print("\nDataset ready for Step 4 (LLM text generation)")
print(f"Columns: {list(df_ranked.columns)}")

In [ ]:
"""Step 4d: Save results with clinical descriptions"""

# Save dataframe with all features, labels, and descriptions
if 'clinical_description' in df_ranked.columns:
    output_csv = "features_with_descriptions.csv"
    df_ranked.to_csv(output_csv, index=False)
    print(f"✓ Saved results to {output_csv}")
    print(f"  Rows: {len(df_ranked)}")
    print(f"  Columns: {list(df_ranked.columns)}")
    
    # Show example
    print(f"\nExample row:")
    row = df_ranked.iloc[0]
    print(f"  Image: {row.get('filename', 'N/A')}")
    print(f"  Important features: {row.get('important_labels', [])[:3]}...")
    print(f"  Description: {row.get('clinical_description', 'N/A')[:150]}...")
else:
    print("⚠ No clinical descriptions found")
    print("  Run Step 4c to generate descriptions first")

In [ ]:
"""Step 4c: Generate descriptions for entire dataset

Add clinical descriptions to the dataframe.
Note: First image loads Mistral-7B model, subsequent images reuse cached model.
"""

# If you have important_features in the dataframe:
if 'important_labels' in df_ranked.columns:
    print("Generating clinical descriptions for dataset...")
    print("Note: First description will load Mistral-7B (~5-10 min on CPU)")
    
    descriptions = []
    for idx, row in df_ranked.iterrows():
        important_features = row.get('important_labels', [])
        
        # Optional: create classification from metadata if available
        classification = None
        # if 'diagnosis' in row:
        #     classification = ClassificationResult(
        #         feature_type=FeatureType.MULTIPLE,
        #         structure=Structure.LINES,
        #         properties=[],
        #         final_class=row['diagnosis'],
        #     )
        
        result = generate_description(
            important_features=important_features if important_features else [],
            classification=classification,
            device=device,
        )
        descriptions.append(result['description'])
        
        if (idx + 1) % 10 == 0:
            print(f"  Generated descriptions for {idx + 1}/{len(df_ranked)} images")
    
    df_ranked['clinical_description'] = descriptions
    print(f"\n✓ Generated {len(descriptions)} clinical descriptions")
    print(f"\nExample description:")
    print(f"  {descriptions[0][:200]}...")
else:
    print("⚠ No 'important_labels' column found")
    print("  Run rank_features() or rank_features_batch() first")

In [ ]:
"""Step 4b: Generate description for a single image (Simple example)

Without importance model - just use extracted features (all of them).
Classification is optional for additional context.
"""

# Example 1: Minimal (just features, no classification)
# image_path = "path/to/image.jpg"
# result = generate_description(
#     important_features=["area:large", "color:красная"],
# )
# print("Clinical description:")
# print(result['description'])

# Example 2: With classification context
# classification = ClassificationResult(
#     feature_type=FeatureType.MULTIPLE,
#     structure=Structure.LINES,
#     properties=["ретикулярный", "симметричный"],
#     final_class="Диспластический невус",
# )
# result = generate_description(
#     important_features=["area:large", "color:красная"],
#     classification=classification,
#     device=device,
# )
# print("Clinical description:")
# print(result['description'])

In [ ]:
"""Step 4a: Generate description for a single image (Full 4-step pipeline)

This function combines all 4 pipeline steps:
1. Extract features
2. Bucket features
3. Rank important features
4. Generate clinical description
"""

def generate_full_pipeline(image_path, importance_model_path=None, classification=None):
    """
    Run complete 4-step pipeline: extract → bucket → rank → generate description
    
    Args:
        image_path: Path to lesion image
        importance_model_path: Optional path to trained importance ranking model
        classification: Optional ClassificationResult for additional context
    
    Returns:
        dict with features, labels, important_features, and clinical description
    """
    # Step 1: Extract features
    extraction = extract_features_and_rank(
        image_path=image_path,
        yolo_weights=YOLO_WEIGHTS,
        unet_weights=UNET_WEIGHTS,
        importance_model_path=importance_model_path,
    )
    features_dict = extraction['features_dict']
    
    # Step 2: Bucket features
    bucketed = bucket_features(features_dict)
    labels = bucketed['labels']
    
    # Step 3: Get important features (from extraction if model was provided)
    important_features = extraction.get('important_features', [])
    
    # Step 4: Generate clinical description using Mistral-7B
    # Note: First call loads the model (~5-10 min on CPU, ~30 sec on GPU)
    result_description = generate_description(
        important_features=important_features if important_features else [],
        classification=classification,
        device=device,
    )
    
    return {
        'features': extraction['features'],
        'features_dict': features_dict,
        'labels': labels,
        'important_features': important_features,
        'description': result_description['description'],
        'model_used': result_description['model'],
    }

# Example usage:
# result = generate_full_pipeline(
#     "test_image.jpg",
#     importance_model_path="importance_checkpoints/best.pt",
#     classification=ClassificationResult(
#         feature_type=FeatureType.MULTIPLE,
#         structure=Structure.LINES,
#         properties=["ретикулярный"],
#         final_class="Диспластический невус",
#     ),
# )
# print("Description:", result['description'])

In [ ]:
"""Step 4: Import text generation modules

Mistral-7B is a lightweight open-source LLM with excellent Russian support.
First call will download the model (~14GB) - takes 5-10 min on CPU, ~30 sec on GPU.
Subsequent calls reuse the cached model (fast).
"""

from generation.description_inference import generate_description
from generation.classification_types import ClassificationResult, Structure, FeatureType

## Stage 4. Генерация клинического описания на основе важных признаков (Mistral-7B)